# Build Task Bank

Downloads a HuggingFace dataset and converts it into task objects defined in `models/task.py`.  
Currently configured for **Summarization** using [EdinburghNLP/xsum](https://huggingface.co/datasets/EdinburghNLP/xsum).

**XSum columns**
| Column | Type | Description |
|--------|------|-------------|
| `document` | string | Full news article (model input) |
| `summary` | string | One-sentence extreme summary (ground truth) |
| `id` | string | BBC article ID |

**Splits:** train (204 045) · validation (11 332) · test (11 334)

## 1. Imports & config

In [ ]:
import json
import random
import sys
from datetime import date
from pathlib import Path
from typing import Literal

import nltk
import textstat
from datasets import load_dataset

# Make models/ importable from data_pipeline/
sys.path.append(str(Path("__file__").resolve().parent.parent))
from models.task import SummarizationTask, Task

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_NAME   = "EdinburghNLP/xsum"
OUTPUT_FILE    = Path("../task_bank/summarization_tasks.jsonl")
SEED           = 42

# How many tasks to sample per split (set to None to use all)
N_TRAIN        = 500
N_VALIDATION   = 100
N_TEST         = 100

random.seed(SEED)

In [ ]:
# Download required nltk data (runs fast if already cached)
nltk.download("punkt",        quiet=True)
nltk.download("punkt_tab",    quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

## 2. Load dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
print(dataset)

In [ ]:
# Sanity check — inspect one example
example = dataset["train"][0]
print("ID       :", example["id"])
print("Summary  :", example["summary"])
print("Document :", example["document"][:300], "...")

## 3. Helper functions

In [ ]:
# POS tags that count as content words for lexical density
_CONTENT_TAGS = {
    "NN", "NNS", "NNP", "NNPS",          # nouns
    "VB", "VBD", "VBG", "VBN", "VBP", "VBZ",  # verbs
    "JJ", "JJR", "JJS",                   # adjectives
    "RB", "RBR", "RBS",                   # adverbs
}


def _compression_ratio_score(document: str, summary: str) -> float:
    """
    doc_words / summary_words — higher means more compression needed → harder.
    Normalised to [0, 1] with a ceiling at ratio 60.
    """
    ratio = len(document.split()) / max(len(summary.split()), 1)
    return min(ratio / 60.0, 1.0)


def _readability_score(document: str) -> float:
    """
    Flesch-Kincaid grade level via textstat — higher grade → harder text.
    Normalised to [0, 1] over a 0–20 grade range.
    """
    grade = textstat.flesch_kincaid_grade(document)
    return min(max(grade, 0.0) / 20.0, 1.0)


def _lexical_density_score(document: str) -> float:
    """
    Ratio of content words to total tokens via nltk POS tagging.
    Higher density = more information per word → harder to summarise.
    Computed on first 1 000 chars for speed; normalised from [0.3, 0.7] to [0, 1].
    """
    tokens = nltk.word_tokenize(document[:1000])
    if not tokens:
        return 0.0
    pos_tags = nltk.pos_tag(tokens)
    density = sum(1 for _, tag in pos_tags if tag in _CONTENT_TAGS) / len(tokens)
    # Shift and scale: 0.3 → 0.0, 0.7 → 1.0
    return min(max((density - 0.3) / 0.4, 0.0), 1.0)


def assign_difficulty(document: str, summary: str) -> Literal["easy", "medium", "hard"]:
    """
    Combined difficulty score (weighted average of three signals):
      40% compression ratio  — how much needs to be distilled
      35% FK readability     — linguistic complexity of the source
      25% lexical density    — information density per word

    Thresholds: score < 0.35 → easy | < 0.65 → medium | else → hard
    """
    score = (
        0.40 * _compression_ratio_score(document, summary)
        + 0.35 * _readability_score(document)
        + 0.25 * _lexical_density_score(document)
    )
    if score < 0.35:
        return "easy"
    if score < 0.65:
        return "medium"
    return "hard"


def infer_domain(document: str) -> str:
    """
    Lightweight keyword-based domain tag.
    XSum is BBC news; we tag the broad topic from the first ~300 chars.
    """
    first = document[:300].lower()
    if any(w in first for w in ("parliament", "minister", "government", "election", "vote", "party", "mp ")):
        return "politics"
    if any(w in first for w in ("police", "murder", "court", "guilty", "sentenced", "crime", "arrested")):
        return "crime"
    if any(w in first for w in ("match", "goal", "league", "footballer", "tennis", "olympic", "cricket")):
        return "sports"
    if any(w in first for w in ("economy", "bank", "market", "shares", "gdp", "inflation", "company", "profit")):
        return "business"
    if any(w in first for w in ("technology", "software", "app", "google", "apple", "ai", "robot", "cyber")):
        return "technology"
    if any(w in first for w in ("hospital", "patient", "nhs", "cancer", "health", "treatment", "drug")):
        return "health"
    return "general"


RUBRIC = (
    "Score the model's summary on three dimensions (1–5 each):\n"
    "1. Faithfulness — does it contain only information from the article? "
    "Penalise hallucinations heavily.\n"
    "2. Informativeness — does it capture the single most important fact?\n"
    "3. Fluency — is it grammatically correct and natural?\n"
    "Return a JSON object: {\"faithfulness\": int, \"informativeness\": int, \"fluency\": int, \"overall\": float}"
)


def row_to_task(row: dict, split: str, index: int) -> SummarizationTask:
    return SummarizationTask(
        task_id    = f"summ_xsum_{split}_{index:06d}",
        difficulty = assign_difficulty(row["document"], row["summary"]),
        domain     = infer_domain(row["document"]),
        input      = row["document"],
        expected   = row["summary"],
        rubric     = RUBRIC,
        metadata   = {
            "source"     : DATASET_NAME,
            "original_id": row["id"],
            "split"      : split,
            "date_added" : str(date.today()),
        },
    )

## 4. Sample & convert

In [ ]:
def sample_split(split_name: str, n: int | None) -> list[dict]:
    split = dataset[split_name]
    indices = list(range(len(split)))
    if n is not None:
        indices = random.sample(indices, min(n, len(indices)))
    tasks = [row_to_task(split[i], split_name, i) for i in indices]
    print(f"{split_name:10s}: {len(tasks)} tasks")
    return [t.model_dump() for t in tasks]


all_tasks = (
    sample_split("train",      N_TRAIN)
    + sample_split("validation", N_VALIDATION)
    + sample_split("test",       N_TEST)
)

print(f"\nTotal tasks: {len(all_tasks)}")

## 5. Inspect difficulty & domain distribution

In [ ]:
from collections import Counter

difficulty_counts = Counter(t["difficulty"] for t in all_tasks)
domain_counts     = Counter(t["domain"]     for t in all_tasks)

print("Difficulty distribution:", dict(difficulty_counts))
print("Domain distribution     :", dict(domain_counts))

## 6. Save to JSONL

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for task in all_tasks:
        f.write(json.dumps(task, ensure_ascii=False) + "\n")

print(f"Saved {len(all_tasks)} tasks → {OUTPUT_FILE.resolve()}")

## 7. Reload & validate round-trip

In [ ]:
from pydantic import TypeAdapter

adapter = TypeAdapter(Task)

loaded = []
with OUTPUT_FILE.open(encoding="utf-8") as f:
    for line in f:
        loaded.append(adapter.validate_json(line))

print(f"Validated {len(loaded)} tasks — all parsed as SummarizationTask:",
      all(isinstance(t, SummarizationTask) for t in loaded))
print("\nSample task:")
print(loaded[0].model_dump_json(indent=2))